In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
Dataset = pd.read_csv('Hybrid_Energy_Site_Samples_PCS_30m.csv')  
Dataset.head()

,system:index,Dist_to_railway,Dist_to_road,Dist_to_tyranss,Elevation,GHI,LULC,River_Dist,Wind,.geo
0,0,39714.628906,8147.200195,32637.005859,411.0,5.760000,2.0,598.977051,5.250,"{""type"":""Point"",""coordinates"":[8.3577885899435..."
1,1,16242.950195,0.000000,156.889343,7.0,5.118325,11.0,0.000000,3.251,"{""type"":""Point"",""coordinates"":[3.1797109827120..."
2,2,92478.523438,0.000000,101954.906250,400.0,6.160000,11.0,3810.678467,6.002,"{""type"":""Point"",""coordinates"":[13.875265934311..."
3,3,161591.093750,1016.463501,129096.289062,272.0,6.050000,7.0,10123.613281,4.120,"{""type"":""Point"",""coordinates"":[9.9207774965204..."
4,4,63809.843750,3229.203613,28574.027344,365.0,6.180000,11.0,0.000000,6.925,"{""type"":""Point"",""coordinates"":[11.946928578230..."


In [3]:
# Dropoing unwanted columns
CleanedDataset = Dataset.drop(columns=[".geo", "system:index"])
CleanedDataset.head()

,Dist_to_railway,Dist_to_road,Dist_to_tyranss,Elevation,GHI,LULC,River_Dist,Wind
0,39714.628906,8147.200195,32637.005859,411.0,5.760000,2.0,598.977051,5.250
1,16242.950195,0.000000,156.889343,7.0,5.118325,11.0,0.000000,3.251
2,92478.523438,0.000000,101954.906250,400.0,6.160000,11.0,3810.678467,6.002
3,161591.093750,1016.463501,129096.289062,272.0,6.050000,7.0,10123.613281,4.120
4,63809.843750,3229.203613,28574.027344,365.0,6.180000,11.0,0.000000,6.925


In [5]:


# --- Classification Functions ---
CleanedDataset_copy = CleanedDataset.copy()
import pandas as pd

# Load your data
df = CleanedDataset_copy

# Define scoring functions
def wind_score(w):
    if w >= 6.0: return 4
    elif 5.0 <= w < 6.0: return 3
    elif 4.0 <= w < 5.0: return 2
    else: return 1

def ghi_score(g):
    if g >= 5.75: return 4
    elif 4.93 <= g < 5.75: return 3
    elif 4.11 <= g < 4.93: return 2
    else: return 1

def lulc_score(l):
    if l in [2, 5, 11]: return 4  # Suitable
    elif l == 3: return 2        # Unsuitable (Forest)
    elif l in [0, 4]: return 1   # Not suitable
    else: return 1

def trans_line_score(d):
    if 250.1 <= d <= 5000: return 4
    elif 5001 <= d <= 10000: return 3
    elif 10001 <= d <= 20000: return 2
    else: return 1

def road_score(d):
    if 500 <= d <= 2500: return 4
    elif 2500 < d <= 5000: return 3
    elif 5000 < d <= 10000: return 2
    else: return 1

def rail_score(d):
    if d >= 280000:
        return 4  # Highly Suitable
    elif 200000 <= d < 280000:
        return 3  # Suitable
    elif 71000 <= d < 200000:
        return 2  # Marginally Suitable
    else:
        return 1  # Unsuitable

def elevation_score(e):
    if e < 250: return 4
    elif e < 500: return 3
    elif e < 1000: return 2
    else: return 1

def river_score(d):
    return 1 if 500 <= d <= 1000 else 0

# Apply scores
df["wind_score"] = df["Wind"].apply(wind_score)
df["ghi_score"] = df["GHI"].apply(ghi_score)
df["lulc_score"] = df["LULC"].apply(lulc_score)
df["trans_score"] = df["Dist_to_tyranss"].apply(trans_line_score)
df["road_score"] = df["Dist_to_road"].apply(road_score)
df["rail_score"] = df["Dist_to_railway"].apply(rail_score)
df["elev_score"] = df["Elevation"].apply(elevation_score)
df["hydrogen_zone"] = df["River_Dist"].apply(river_score)

# Weighted total score
df["total_score"] = (
    df["ghi_score"] * 0.25 +
    df["wind_score"] * 0.25 +
    df["lulc_score"] * 0.20 +
    df["trans_score"] * 0.10 +
    df["road_score"] * 0.07 +
    df["rail_score"] * 0.05 +
    df["elev_score"] * 0.05
)

# Mapping dictionary for later reference
suitability_mapping = {
    3: "Highly Suitable",
    2: "Suitable",
    1: "Marginally Suitable",
    0: "Unsuitable"
}

# Label suitability as numeric codes
def label_suitability(score):
    if score >= 3.4:
        return 3  # Highly Suitable
    elif score >= 2.9:
        return 2  # Suitable
    elif score >= 2.4:
        return 1  # Marginally Suitable
    else:
        return 0  # Unsuitable

df["Suitability"] = df["total_score"].apply(label_suitability)



In [6]:
CleanedDataset_copy.head(1000)

,Dist_to_railway,Dist_to_road,Dist_to_tyranss,Elevation,GHI,LULC,River_Dist,Wind,wind_score,ghi_score,lulc_score,trans_score,road_score,rail_score,elev_score,hydrogen_zone,total_score,Suitability
0,39714.628906,8147.200195,32637.005859,411.0,5.760000,2.0,598.977051,5.250,3,4,4,1,2,1,3,1,2.99,2
1,16242.950195,0.000000,156.889343,7.0,5.118325,11.0,0.000000,3.251,1,3,4,1,1,1,4,0,2.22,0
2,92478.523438,0.000000,101954.906250,400.0,6.160000,11.0,3810.678467,6.002,4,4,4,1,1,2,3,0,3.22,2
3,161591.093750,1016.463501,129096.289062,272.0,6.050000,7.0,10123.613281,4.120,2,4,1,1,4,2,3,0,2.33,0
4,63809.843750,3229.203613,28574.027344,365.0,6.180000,11.0,0.000000,6.925,4,4,4,1,3,1,3,0,3.31,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,44228.679688,436.759644,37223.804688,317.0,5.680000,2.0,15007.632812,4.230,2,3,4,1,1,1,3,0,2.42,1
996,19769.134766,7582.682129,32277.058594,82.0,5.760000,11.0,2161.131592,5.250,3,4,4,1,2,1,4,0,3.04,2
997,69994.093750,0.000000,131002.671875,343.0,5.830000,11.0,4435.962891,6.912,4,4,4,1,1,1,3,0,3.17,2
998,1167.346924,1041.458130,38452.664062,854.0,6.090000,11.0,14045.658203,6.825,4,4,4,1,4,1,2,0,3.33,2


In [7]:

suitability_counts = df['Suitability'].value_counts()

# Display the counts
print("Suitability Class Counts:")
print(suitability_counts)



Suitability Class Counts:
Suitability
2    15131
1     6773
3     4091
0     4004
Name: count, dtype: int64


In [8]:
X = CleanedDataset_copy[['GHI', 'Dist_to_railway', 'Dist_to_road', 'Dist_to_tyranss', 'Elevation', 'LULC', 'Wind']]
y = CleanedDataset_copy['Suitability']  

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=50)

forest = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
forest.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
pred_forest = forest.predict(X_test)

from sklearn.metrics import accuracy_score

# Accuracy
accuracy = accuracy_score(y_test, pred_forest)
print (f"Accuracy: {accuracy:.2f}")

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, pred_forest))

# Detailed Classification Report
print("Classification Report:")
print(classification_report(y_test, pred_forest))


Accuracy: 0.97
Confusion Matrix:
[[1180   58    5    0]
 [  28 1934   54    3]
 [   0   34 4443   16]
 [   0    1   39 1205]]
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.95      0.96      1243
           1       0.95      0.96      0.96      2019
           2       0.98      0.99      0.98      4493
           3       0.98      0.97      0.98      1245

    accuracy                           0.97      9000
   macro avg       0.97      0.97      0.97      9000
weighted avg       0.97      0.97      0.97      9000



In [10]:
import joblib

# Save the model
joblib.dump(forest, "suitability_model.pkl")

['suitability_model.pkl']